In [ ]:
knitr::opts_chunk$set(echo = TRUE)

Wir probieren das mal mit einer ganz einfachen Poisson-Regression ohne Offset, aber mit der Textlänge als Prädiktor. Dazu brauchen wir erst mal einen simulierten Datensatz. Diesem Datensatz fügen wir 5 Spalten von simulierten Abkürzungsanzahlen hinzu:

* **Zufällig** basierend auf einer Poisson-Verteilung.
* Schwach **positiv** in Abhängigkeit der Textlänge.
* Stark **positiv** in Abhängigkeit der Textlänge.
* Schwach **negativ** in Abhängigkeit der Textlänge.
* Stark **negativ** in Abhängigkeit der Textlänge.

Auch die simulierten Zusammenhänge basieren dabei auf einer Poisson-Verteilung, wir manipulieren aber den lambda-Parameter entsprechend. Los gehts...

In [ ]:
library(dplyr)

n_texts <- 100 # Anzahl Texte
# Globale Variablen für die Stärke der schwachen und starken Zusammenhänge
weak_effect_factor <- 0.02
strong_effect_factor <- 0.08

set.seed(667)

Der Basis-Dataframe mit der zufälligen Anzahl Abkürzungen:

In [ ]:
texts <- data.frame(text_id = 1:n_texts,
                    n_tokens = rpois(n_texts, lambda = 50), # Anzahl Tokens
                    n_abbr_rnd = rpois(n_texts, lambda = 2)) # Anzahl Abkürzungen (zufällig)

Jetzt fügen wir nach und nach die Spalten mit den simulierten Effekten hinzu.

In [ ]:
# Anzahl Abkürzungen (schwach positiver Effekt der Textlänge)
texts$n_abbr_posweak <- rpois(n_texts, lambda = exp(weak_effect_factor * texts$n_token - 2))
# Anzahl Abkürzungen (stark positiver Effekt der Textlänge)
texts$n_abbr_posstrong <- rpois(n_texts, lambda = exp(strong_effect_factor * texts$n_token - 5))
# Anzahl Abkürzungen (schwach negativer Effekt der Textlänge)
texts$n_abbr_negweak <- rpois(n_texts, lambda = exp(-weak_effect_factor * texts$n_token + 2))
# Anzahl Abkürzungen (stark negativer Effekt der Textlänge)
texts$n_abbr_negstrong <- rpois(n_texts, lambda = exp(-strong_effect_factor * texts$n_token + 5))

Das `exp()` ist nur dafür da, dass der lambda-Parameter immer positiv bleibt. Die Additionen/Subtraktionen hinten sind nur dafür da, dass halbwegs plausible Werte entstehen.

Wie sieht das jetzt aus?

In [ ]:
head(texts)

Ja, geht so. Evtl. könnte man noch mit den Additionen/Subtraktionen am Ende spielen, damit die Werte noch plausibler in Abhängigkeit der Effektrichtung werden, aber es ist erstmal in Ordnung so, denke ich.

Jetzt rechnen wir für alle 5 Effekte einzelne ganz einfache Poisson-Modelle ohne Offset, sondern mit der Textlänge als Prädiktor.

In [ ]:
pois_mod_rnd <- glm(n_abbr_rnd ~ n_tokens, data = texts, family = "poisson")
pois_mod_posweak <- glm(n_abbr_posweak ~ n_tokens, data = texts, family = "poisson")
pois_mod_posstrong <- glm(n_abbr_posstrong ~ n_tokens, data = texts, family = "poisson")
pois_mod_negweak <- glm(n_abbr_negweak ~ n_tokens, data = texts, family = "poisson")
pois_mod_negstrong <- glm(n_abbr_negstrong ~ n_tokens, data = texts, family = "poisson")

Schauen wir mal rein...

In [ ]:
summary(pois_mod_rnd)
summary(pois_mod_posweak)
summary(pois_mod_posstrong)
summary(pois_mod_negweak)
summary(pois_mod_negstrong)

Okay, mit diesem Seed (667) sehen wir folgendes:

* kein Effekt für komplett zufällig
* kein Effekt für schwach positiv
* klarer Effekt für stark positiv
* klarer Effekt für schwach negativ
* klarer Effekt für stark negativ

Aber wie verhält sich das für andere Seeds? Ist das so stabil? Um das herauszufinden, bauen wir uns eine Funktion, die den kompletten Ablauf bis hierher repliziert und uns als Ergebnis einen Dataframe zurückgibt, der nur die p-Werte und Estimates enthält.

In [ ]:
do_test <- function (n_texts, test_no = NA) {
  texts <- data.frame(text_id = 1:n_texts,
                      n_tokens = rpois(n_texts, lambda = 50), # Anzahl Tokens
                      n_abbr_rnd = rpois(n_texts, lambda = 2)) # Anzahl Abkürzungen (zufällig)
  texts$n_abbr_posweak <- rpois(n_texts, lambda = exp(weak_effect_factor * texts$n_tokens - 2))
  texts$n_abbr_posstrong <- rpois(n_texts, lambda = exp(strong_effect_factor * texts$n_tokens - 5))
  texts$n_abbr_negweak <- rpois(n_texts, lambda = exp(-weak_effect_factor * texts$n_tokens + 2))
  texts$n_abbr_negstrong <- rpois(n_texts, lambda = exp(-strong_effect_factor * texts$n_tokens + 5))

  pois_mod_rnd <- glm(n_abbr_rnd ~ n_tokens, data = texts, family = "poisson")
  pois_mod_posweak <- glm(n_abbr_posweak ~ n_tokens, data = texts, family = "poisson")
  pois_mod_posstrong <- glm(n_abbr_posstrong ~ n_tokens, data = texts, family = "poisson")
  pois_mod_negweak <- glm(n_abbr_negweak ~ n_tokens, data = texts, family = "poisson")
  pois_mod_negstrong <- glm(n_abbr_negstrong ~ n_tokens, data = texts, family = "poisson")
  # Hier werden die p-Werte extrahiert und in einem Dataframe zurückgegeben:
  data.frame(test_no = test_no,
             n_texts = n_texts,
             est_rnd = summary(pois_mod_rnd)$coefficients[2,1],
             p_rnd = summary(pois_mod_rnd)$coefficients[2,4],
             est_posweak = summary(pois_mod_posweak)$coefficients[2,1],
             p_posweak = summary(pois_mod_posweak)$coefficients[2,4],
             est_posstrong = summary(pois_mod_posstrong)$coefficients[2,1],
             p_posstrong = summary(pois_mod_posstrong)$coefficients[2,4],
             est_negweak = summary(pois_mod_negweak)$coefficients[2,1],
             p_negweak = summary(pois_mod_negweak)$coefficients[2,4],
             est_negstrong = summary(pois_mod_negstrong)$coefficients[2,1],
             p_negstrong = summary(pois_mod_negstrong)$coefficients[2,4])
}

Jetzt lassen wir den Seed wieder frei variieren und lassen das Ganze 5000x laufen. Jedes Ergebnis speichern wir in einer Liste, die wir später in einen Dataframe umwandeln. Als Anzahl Texte benutze ich jetzt erst mal immer 100. Es ist zu erwarten, dass die Regressionen eher einen Effekt anzeigen, wenn wir mehr Texte verwenden (aber **nicht** für die zufällige Anzahl Abkürzungen), weil die Stichprobe größer ist.

In [ ]:
set.seed(NULL)
test_results <- list()
for (i in 1:5000) { # Anzahl Durchläufe
  if (i %% 1000 == 0) cat("Durchlauf", i, "\n")
  new_test <- do_test(n_texts = 100, test_no = i)
  test_results[[i]] <- new_test
}
test_results <- bind_rows(test_results)

Alles klar, jetzt schauen wir uns an, wie oft die einzelnen Effekte signifikant waren und ob für die simulierten Effekte das Vorzeichen des Estimates auch korrekt ist. Es ist zu erwarten, dass auch die Regressionen für die zufällige Abkürzungsanzahl in einigen Fällen signifikant sind. Das sollten aber ungefähr 5% sein, denn das ist die Irrtumswahrscheinlichkeit. Je mehr dieser Tests wir laufen lassen, desto näher sollte dieser Wert bei 5% liegen.

In [ ]:
test_results %>%
  summarise(p_sig_rnd = sum(p_rnd < 0.05) / n(),
            p_sig_posweak = sum(p_posweak < 0.05 & est_posweak > 0) / n(),
            p_sig_posstrong = sum(p_posstrong < 0.05 & est_posstrong > 0) / n(),
            p_sig_negweak = sum(p_negweak < 0.05 & est_negweak < 0) / n(),
            p_sig_negstrong = sum(p_negstrong < 0.05 & est_negstrong < 0) / n())

Wir sehen, dass das funktioniert hat: Wenn kein Effekt vorliegt, wird in ca. 5% aller Fälle einer gefunden (= Irrtumswahrscheinlichkeit), in allen anderen Fällen liegen die Zahlen über 5%, bei stärkeren Effekten deutlich höher.

Probieren wir das nochmal mit einer größeren Anzahl Texten...

In [ ]:
test_results <- list()
for (i in 1:5000) { # Anzahl Durchläufe
  if (i %% 1000 == 0) cat("Durchlauf", i, "\n")
  new_test <- do_test(n_texts = 1000, test_no = i) # <- Hier habe ich n_texts auf 1000 gesetzt.
  test_results[[i]] <- new_test
}
test_results <- bind_rows(test_results)

In [ ]:
test_results %>%
  summarise(p_sig_rnd = sum(p_rnd < 0.05) / n(),
            p_sig_posweak = sum(p_posweak < 0.05 & est_posweak > 0) / n(),
            p_sig_posstrong = sum(p_posstrong < 0.05 & est_posstrong > 0) / n(),
            p_sig_negweak = sum(p_negweak < 0.05 & est_negweak < 0) / n(),
            p_sig_negstrong = sum(p_negstrong < 0.05 & est_negstrong < 0) / n())

Auch das funktioniert entsprechend den Erwartungen: Immer noch ca. 5% für eine zufällige Anzahl Abkürzungen, aber deutlich höhere Werte für die simulierten Effekte als für 100 Texte. 